# 条件 Flow Matching：给定类别标签生成图像

**速度场** `v_θ(x, t, y)`：输入含噪图像 `x_t`、连续时间 `t∈[0,1]`、类别 `y`，输出与图像同形状的向量场。

**训练（CFM）**：`x_t = (1-t)x_0 + t x_1`，`x_0∼N(0,I)`，`x_1` 为数据图像；目标 `u = x_1 - x_0`；最小化 `||v_θ(x_t,t,y) - u||²`。

**采样**：采样 `x_0∼N(0,I)`，固定 `y`，用欧拉法积分 `dx/dt = v_θ(x,t,y)` 到 `t=1`。

数据集用下方 **占位 Dataset**；你找到数据后只替换 `Dataset` / `DataLoader`，保持图像 **`[-1,1]`** 与 `NUM_CLASSES` 一致即可。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 超参数

In [ ]:
IMAGE_SIZE = 32
IN_CHANNELS = 3
NUM_CLASSES = 10

BASE_CH = 64
TIME_DIM = 256
CLASS_EMB_DIM = 256

N_STEPS_ODE = 50
BATCH_SIZE = 32
LR = 2e-4
TRAIN_STEPS = 2000

## 模型：时间嵌入 + 类别嵌入 + U-Net 速度场

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        f = math.log(10000) / max(half - 1, 1)
        freqs = torch.exp(torch.arange(half, device=t.device, dtype=t.dtype) * -f)
        args = t.unsqueeze(1) * freqs.unsqueeze(0) * 1000.0
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2:
            emb = F.pad(emb, (0, 1))
        return emb


class ConditionalResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, class_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        cond_dim = time_dim + class_dim
        self.cond_proj = nn.Linear(cond_dim, out_ch * 2)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond_vec):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        h = self.norm2(h)
        scale, shift = self.cond_proj(cond_vec).chunk(2, dim=1)
        scale = scale.unsqueeze(-1).unsqueeze(-1)
        shift = shift.unsqueeze(-1).unsqueeze(-1)
        h = h * (1 + scale) + shift
        h = F.silu(h)
        h = self.conv2(h)
        return h + self.skip(x)


class ConditionalVelocityUNet(nn.Module):
    """v(x_t, t, y)，输出 (B,C,H,W)。"""

    def __init__(
        self,
        in_channels=3,
        base_ch=64,
        time_dim=256,
        num_classes=10,
        class_emb_dim=256,
    ):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbed(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.class_embed = nn.Embedding(num_classes, class_emb_dim)

        self.conv_in = nn.Conv2d(in_channels, base_ch, 3, padding=1)

        self.down1 = ConditionalResBlock(base_ch, base_ch, time_dim, class_emb_dim)
        self.down_sample1 = nn.Conv2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.down2 = ConditionalResBlock(base_ch, base_ch * 2, time_dim, class_emb_dim)
        self.down_sample2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.down3 = ConditionalResBlock(base_ch * 2, base_ch * 4, time_dim, class_emb_dim)

        self.mid = ConditionalResBlock(base_ch * 4, base_ch * 4, time_dim, class_emb_dim)

        self.up3 = ConditionalResBlock(base_ch * 8, base_ch * 4, time_dim, class_emb_dim)
        self.up_sample2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        self.up2 = ConditionalResBlock(base_ch * 6, base_ch * 2, time_dim, class_emb_dim)
        self.up_sample1 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.up1 = ConditionalResBlock(base_ch * 3, base_ch, time_dim, class_emb_dim)

        self.conv_out = nn.Conv2d(base_ch, in_channels, 3, padding=1)

    def forward(self, x, t, y):
        if t.dim() > 1:
            t = t.squeeze(-1)
        te = self.time_embed(t)
        ce = self.class_embed(y)
        cond_vec = torch.cat([te, ce], dim=1)

        h0 = self.conv_in(x)
        h1 = self.down1(h0, cond_vec)
        h1d = self.down_sample1(h1)
        h2 = self.down2(h1d, cond_vec)
        h2d = self.down_sample2(h2)
        h3 = self.down3(h2d, cond_vec)

        m = self.mid(h3, cond_vec)

        u3 = self.up3(torch.cat([m, h3], dim=1), cond_vec)
        u3 = self.up_sample2(u3)
        u2 = self.up2(torch.cat([u3, h2], dim=1), cond_vec)
        u2 = self.up_sample1(u2)
        u1 = self.up1(torch.cat([u2, h1], dim=1), cond_vec)

        return self.conv_out(u1)

## 占位数据（可替换）

In [ ]:
class PlaceholderLabelImageDataset(Dataset):
    """随机图 + 标签；接入真实数据时保持 image ∈ [-1,1]，label ∈ {0,…,C-1}。"""

    def __init__(self, n=10000, size=32, c=3, num_classes=10):
        self.n = n
        self.size = size
        self.c = c
        self.num_classes = num_classes

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        y = torch.randint(0, self.num_classes, (1,)).long().item()
        g = torch.randn(self.c, self.size, self.size)
        bias = torch.tensor(
            [math.cos(y), math.sin(y), y / max(self.num_classes - 1, 1)], dtype=g.dtype
        ).view(3, 1, 1)
        x = (g * 0.3 + bias).clamp(-1.0, 1.0)
        return x, y

## CFM 损失 + 条件采样

In [ ]:
def cfm_loss_velocity(model, x1, y):
    b = x1.shape[0]
    device = x1.device
    x0 = torch.randn_like(x1)
    t = torch.rand(b, device=device)
    tt = t.view(-1, 1, 1, 1)
    x_t = (1.0 - tt) * x0 + tt * x1
    target = x1 - x0
    pred = model(x_t, t, y)
    return F.mse_loss(pred, target)


@torch.no_grad()
def sample_conditional(model, labels, image_shape, n_steps=50, device=device):
    model.eval()
    b = labels.shape[0]
    x = torch.randn(b, *image_shape, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((b,), i * dt, device=device)
        v = model(x, t, labels)
        x = x + v * dt
    return x.clamp(-1.0, 1.0)

## 训练

In [ ]:
ds = PlaceholderLabelImageDataset(
    n=8000, size=IMAGE_SIZE, c=IN_CHANNELS, num_classes=NUM_CLASSES
)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

model = ConditionalVelocityUNet(
    in_channels=IN_CHANNELS,
    base_ch=BASE_CH,
    time_dim=TIME_DIM,
    num_classes=NUM_CLASSES,
    class_emb_dim=CLASS_EMB_DIM,
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR)

loss_hist = []
step = 0
while step < TRAIN_STEPS:
    for x1, y in loader:
        x1 = x1.to(device)
        y = y.to(device)
        loss = cfm_loss_velocity(model, x1, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())
        step += 1
        if step % 200 == 0:
            print(f"step {step}/{TRAIN_STEPS}  loss={loss.item():.6f}")
        if step >= TRAIN_STEPS:
            break

plt.figure(figsize=(5, 2.5))
plt.plot(loss_hist)
plt.xlabel("step")
plt.ylabel("MSE")
plt.title("CFM loss")
plt.tight_layout()
plt.show()

## 按标签生成并可视化

In [ ]:
n_show = min(10, NUM_CLASSES)
labels = torch.arange(n_show, device=device)
gen = sample_conditional(
    model,
    labels,
    (IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE),
    n_steps=N_STEPS_ODE,
    device=device,
)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        break
    im = gen[i].cpu().permute(1, 2, 0).numpy()
    im = (im + 1) / 2.0
    ax.imshow(im.clip(0, 1))
    ax.set_title(f"y={labels[i].item()}")
    ax.axis("off")
plt.suptitle("Conditional generation (placeholder data)")
plt.tight_layout()
plt.show()

### 接入你自己的数据集

- 图像归一化到 **`[-1, 1]`**（例如 `x = x / 255 * 2 - 1`）。
- `NUM_CLASSES`、`IMAGE_SIZE` 与数据一致；若边长不是 32，本 U-Net 为两次 stride-2 下采样，**高宽需能被 4 整除**（或先 resize 到 32/64）。
- 生成时：`labels = torch.tensor([3, 3, 7, ...], device=device)` 指定要的类别。